<a href="https://colab.research.google.com/github/Rayudu-Somisetty/deep_learning_lab_tasks/blob/main/sigmoid_func_with_12_neurons_or_hidden_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#sigmoid func with 12 neurons or hiddenlayer

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load dataset
data = pd.read_csv("data.csv")

# Clean and drop useless columns
data.columns = data.columns.str.strip()
data = data.drop(columns=["Unnamed: 32"], errors="ignore")
data = data.drop(columns=["id"], errors="ignore")

# Target: diagnosis (B=0, M=1)
data["diagnosis"] = data["diagnosis"].map({"B": 0, "M": 1})

# Split X and y
X = data.drop(columns=["diagnosis"]).values
y = data["diagnosis"].values.reshape(-1, 1)

# Split (IMPORTANT: use 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

# Neural Network (12 hidden neurons)
class NeuralNetwork:
    def __init__(self, input_size, hidden_size=12, lr=0.01, epochs=5000):
        self.lr = lr
        self.epochs = epochs

        # Xavier initialization (better than random)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, 1) * np.sqrt(1 / hidden_size)
        self.b2 = np.zeros((1, 1))

        self.loss_history = []

    def fit(self, X, y):
        n = X.shape[0]

        for epoch in range(self.epochs):
            # Forward
            z1 = X @ self.W1 + self.b1
            a1 = relu(z1)

            z2 = a1 @ self.W2 + self.b2
            a2 = sigmoid(z2)

            # Binary Cross Entropy loss
            eps = 1e-9
            loss = -np.mean(y * np.log(a2 + eps) + (1 - y) * np.log(1 - a2 + eps))
            self.loss_history.append(loss)

            # Backprop
            dz2 = (a2 - y)  # BCE + sigmoid derivative combined
            dW2 = (a1.T @ dz2) / n
            db2 = np.sum(dz2, axis=0, keepdims=True) / n

            dz1 = (dz2 @ self.W2.T) * relu_derivative(z1)
            dW1 = (X.T @ dz1) / n
            db1 = np.sum(dz1, axis=0, keepdims=True) / n

            # Update
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1

    def predict(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = sigmoid(z2)
        return (a2 >= 0.5).astype(int)

# Train
nn = NeuralNetwork(input_size=X_train.shape[1], hidden_size=12, lr=0.01, epochs=5000)
nn.fit(X_train, y_train)

# Accuracy
train_acc = accuracy_score(y_train, nn.predict(X_train))
test_acc = accuracy_score(y_test, nn.predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)


Train Accuracy: 0.989010989010989
Test Accuracy : 0.9824561403508771
